# Lung Cancer Dataset: EDA and Machine Learning

This notebook explores patterns related to lung cancer survival and tests a simple logistic regression model. The main data-processing steps are stored as reusable functions in `src/analysis.py`, which makes the project easier to test and reproduce.

## 1. Import Libraries and Functions

Import plotting tools and the reusable functions used throughout the analysis.

In [ ]:
import matplotlib.pyplot as plt

from src.analysis import (
    calculate_stage_survival,
    filter_stages,
    load_data,
    preprocess_data,
    smoking_counts_by_stage,
    train_and_evaluate_model,
)

## 2. Load the Dataset

`load_data()` reads the CSV file and checks that the columns required by this analysis are present. This gives a clear error if the wrong dataset is used.

In [ ]:
df = load_data("lung_cancer_dataset.csv")
df.head()

## 3. Inspect the Data

Review the dataset's size, data types, summary statistics, duplicate rows, unique patient IDs, and missing values before transforming it.

In [ ]:
# Number of rows and columns
df.shape

In [ ]:
# Data types and non-null values
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

In [ ]:
# Check for duplicate rows
df.duplicated().sum()

In [ ]:
# Confirm that patient IDs are unique
df["Patient_ID"].nunique() == len(df)

In [ ]:
# Check for missing values
df.isna().sum()

## 4. Prepare Variables for Analysis

`preprocess_data()` converts `Survived` from Yes/No to 1/0 and cancer stages from Stage I-IV to 1-4. It returns a copy, so the original DataFrame is not modified accidentally.

In [ ]:
df = preprocess_data(df)
df[["Survived", "Survived_num", "Cancer_Stage", "Stage_num"]].head()

## 5. Survival Rate by Cancer Stage

Group patients by cancer stage and compare both the percentage who survived and the number of patients in each group.

In [ ]:
stage_survival = calculate_stage_survival(df)
stage_survival

**Finding:** Survival rate decreases sharply as cancer stage advances. Stage I has the highest survival rate, while Stage IV has the lowest.

In [ ]:
ax = stage_survival["Survival_Rate"].plot(
    kind="bar",
    figsize=(5, 3.5),
    width=0.65,
)

plt.xlabel("Cancer Stage")
plt.ylabel("Survival Rate (%)")
plt.title("Survival Rate by Cancer Stage")
plt.ylim(0, 80)
plt.xticks(rotation=0)
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
plt.tight_layout()
plt.show()

## 6. Compare Smoking Status in Stage I and Stage IV

These stages had the largest difference in survival rates. The comparison below describes smoking-status patterns within the two groups; it does not establish that smoking status caused the stage difference.

In [ ]:
stage_1_4 = filter_stages(df)
smoking_by_stage = smoking_counts_by_stage(stage_1_4)
smoking_by_stage

**Finding:** Stage IV has more current smokers, while Stage I has more patients who never smoked. This is an association in the dataset, not evidence of causation.

## 7. Tumor Size vs. Survival Months

Use a scatter plot and correlation to examine the direction and strength of the linear relationship between tumor size and survival time.

In [ ]:
df.plot(
    kind="scatter",
    x="Tumor_Size_cm",
    y="Survival_Months",
    figsize=(6, 4),
)

plt.xlabel("Tumor Size (cm)")
plt.ylabel("Survival Months")
plt.title("Tumor Size vs. Survival Months")
plt.tight_layout()
plt.show()

In [ ]:
tumor_survival_correlation = df["Tumor_Size_cm"].corr(df["Survival_Months"])
tumor_survival_correlation

**Finding:** The original dataset produced a correlation of approximately -0.69. This indicates a fairly strong negative relationship: larger tumors tend to be associated with shorter survival times in this dataset.

## 8. Machine Learning: Logistic Regression

**Question:** Can cancer stage, tumor size, and age be used to predict whether a patient survives?

The data is divided into 80% training data and 20% test data. `random_state=42` makes the split reproducible, while stratification preserves the survival-class proportions in both sets.

In [ ]:
(
    model,
    y_pred,
    accuracy,
    X_train,
    X_test,
    y_train,
    y_test,
) = train_and_evaluate_model(df)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
# Preview the first 10 predicted survival outcomes
# 1 means survived and 0 means did not survive.
y_pred[:10]

**Model interpretation:** Accuracy is the percentage of test observations the model classified correctly. The original notebook achieved approximately 75% accuracy. This is an initial model using only three predictors, so the result should not be interpreted as a clinical tool.

## 9. Reproducibility and Testing

The functions used above are tested in the `tests/` folder:

- Unit tests validate data loading, preprocessing, summaries, filtering, and model output.
- An edge-case test confirms that a dataset with missing required columns is rejected.
- A system test runs the complete workflow from a temporary CSV file through model evaluation.
- GitHub Actions runs all tests automatically after each push or pull request.

Run the tests from the project root with:

```bash
pytest -v
```